# 09 Memory-Anchored OT Matching W1

역할: clean normal에서 prototype별 patch-bag memory와 prototype-level spatial graph를 구축하고, clean self-probe 한 장을 raw DINO patch query로 넣어 memory-anchored unbalanced OT matching이 합리적으로 낮은 anomaly decomposition을 내는지 확인한다.

W1 범위에서는 anomaly AUROC, shifted/anomaly batch evaluation, learnable head, instance-level graph를 구현하지 않는다.


## Cell Role: Repo Setup

Colab에서는 `/content/ReGraM` checkout을 사용하고, 로컬에서는 현재 repo root를 찾는다. W1 모듈은 `src/stage1_adapter/memory_bank.py`와 `src/stage1_adapter/memory_ot_matcher.py`에 있다.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_URL = 'https://github.com/outSeop/ReGraM.git'


def is_regram_root(path: Path) -> bool:
    return (
        (path / 'experiments' / 'validation' / 'condition_shift_baseline').exists()
        and (path / 'manifests').exists()
    )


colab_checkout = Path('/content/ReGraM')
if Path('/content').exists():
    if (colab_checkout / '.git').exists():
        subprocess.run(['git', '-C', str(colab_checkout), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(
            [
                'git', '-C', str(colab_checkout), 'checkout', 'FETCH_HEAD', '--',
                'experiments/validation/condition_shift_baseline/src',
                'experiments/validation/condition_shift_baseline/notebook/09_memory_anchored_ot_matching.ipynb',
            ],
            check=True,
        )
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(colab_checkout)], check=True)

cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((candidate.resolve() for candidate in repo_candidates if candidate.exists() and is_regram_root(candidate)), None)
if REPO_ROOT is None:
    raise FileNotFoundError('ReGraM checkout not found')

EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
SRC_ROOT = EXP_ROOT / 'src'
for import_root in [SRC_ROOT, SRC_ROOT / 'orchestration']:
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Cell Role: Settings And Data Readiness

W1은 `breakfast_box` clean normal 소수 샘플만 사용한다. `PROBE_SAMPLE_ID`는 memory source에서 제외해 clean self-probe query로 사용한다.


In [ ]:
CATEGORY = 'breakfast_box'
MEMORY_SAMPLE_IDS = ['000.png', '001.png', '002.png', '004.png', '005.png']
PROBE_SAMPLE_ID = '003.png'
FEATURE_BACKBONE = 'dinov2_vits14'
IMAGE_SIZE = 224
MAX_PROTOTYPES = 8
MIN_PAIRS_FOR_SPATIAL_EDGE = 3

RAW_LOCO_ROW_ROOT = REPO_ROOT / 'data' / 'row'
RAW_LOCO_ROOT = RAW_LOCO_ROW_ROOT / 'mvtec_loco_anomaly_detection'
GROUNDING_MASK_ROOT = REPO_ROOT / 'external' / 'UniVAD' / 'masks' / 'mvtec_loco_caption'
PROBE_ROOT = EXP_ROOT / 'reports' / 'memory_anchored_ot_matching'
MEMORY_JSON = PROBE_ROOT / f'{CATEGORY}_w1_memory_bank_summary.json'
SELF_PROBE_JSON = PROBE_ROOT / f'{CATEGORY}_w1_clean_self_probe.json'
PROBE_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_RAW_TAR = Path('/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz')
DRIVE_MASK_ROOT = Path('/content/drive/MyDrive/ReGraM/univad_masks/mvtec_loco_caption')


def mount_drive_if_available() -> None:
    if not Path('/content').exists() or Path('/content/drive/MyDrive').exists():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {type(exc).__name__}: {exc}')


def normalize_dataset_root(parent: Path, dataset_name: str) -> Path:
    direct = parent / dataset_name
    nested_candidates = [parent / 'content' / dataset_name, parent / 'data' / 'row' / dataset_name]
    if direct.exists():
        return direct
    for candidate in nested_candidates:
        if candidate.exists():
            direct.parent.mkdir(parents=True, exist_ok=True)
            print(f'normalize dataset root: {candidate} -> {direct}')
            shutil.move(str(candidate), str(direct))
            return direct
    return direct


def restore_raw_loco_from_drive_if_needed() -> None:
    global RAW_LOCO_ROOT
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')
    if (RAW_LOCO_ROOT / CATEGORY / 'test' / 'good').exists():
        return
    mount_drive_if_available()
    if not DRIVE_RAW_TAR.exists():
        print(f'raw LOCO Drive tar not found: {DRIVE_RAW_TAR}')
        return
    RAW_LOCO_ROW_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'extract raw LOCO from Drive: {DRIVE_RAW_TAR} -> {RAW_LOCO_ROW_ROOT}')
    subprocess.run(['tar', '-xf', str(DRIVE_RAW_TAR), '-C', str(RAW_LOCO_ROW_ROOT)], check=True)
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')


def clean_image_path(image_id: str) -> Path:
    return RAW_LOCO_ROOT / CATEGORY / 'test' / 'good' / image_id


def clean_mask_path(image_id: str) -> Path:
    return GROUNDING_MASK_ROOT / CATEGORY / 'test' / 'good' / Path(image_id).stem / 'grounding_mask.png'


def restore_clean_masks_from_drive_if_needed(image_ids: list[str]) -> None:
    missing = [image_id for image_id in image_ids if not clean_mask_path(image_id).exists()]
    if not missing:
        return
    mount_drive_if_available()
    category_drive_root = DRIVE_MASK_ROOT / CATEGORY
    category_session_root = GROUNDING_MASK_ROOT / CATEGORY
    if category_drive_root.exists() and not category_session_root.exists():
        category_session_root.parent.mkdir(parents=True, exist_ok=True)
        print(f'restore masks category from Drive: {category_drive_root} -> {category_session_root}')
        shutil.copytree(category_drive_root, category_session_root, dirs_exist_ok=True)
    for image_id in missing:
        session_mask = clean_mask_path(image_id)
        drive_candidates = [
            category_drive_root / 'test' / 'good' / Path(image_id).stem / 'grounding_mask.png',
            category_drive_root / Path(image_id).stem / 'grounding_mask.png',
        ]
        drive_mask = next((candidate for candidate in drive_candidates if candidate.exists()), None)
        if drive_mask is None:
            continue
        session_mask.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_mask, session_mask)


restore_raw_loco_from_drive_if_needed()
all_required_ids = MEMORY_SAMPLE_IDS + [PROBE_SAMPLE_ID]
restore_clean_masks_from_drive_if_needed(all_required_ids)
missing = {
    image_id: {'image': clean_image_path(image_id), 'mask': clean_mask_path(image_id)}
    for image_id in all_required_ids
    if not clean_image_path(image_id).exists() or not clean_mask_path(image_id).exists()
}
if missing:
    raise FileNotFoundError(f'Missing W1 clean image/mask inputs: {missing}')

settings_df = pd.DataFrame([
    {
        'category': CATEGORY,
        'memory_sample_ids': ', '.join(MEMORY_SAMPLE_IDS),
        'probe_sample_id': PROBE_SAMPLE_ID,
        'feature_backbone': FEATURE_BACKBONE,
        'image_size': IMAGE_SIZE,
        'max_prototypes': MAX_PROTOTYPES,
        'memory_json': str(MEMORY_JSON),
        'self_probe_json': str(SELF_PROBE_JSON),
    }
])
display(settings_df)


## Cell Role: DINO And Dependency Setup

DINOv2 patch feature extractor를 준비한다. POT가 없으면 설치 후 런타임 재시작 없이 import를 다시 시도한다.


In [ ]:
import importlib.util
import numpy as np
import torch
from PIL import Image

if importlib.util.find_spec('ot') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pot'], check=True)

import ot  # noqa: F401

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

dino_model = torch.hub.load('facebookresearch/dinov2', FEATURE_BACKBONE).to(device).eval()
dino_mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32, device=device).view(1, 3, 1, 1)
dino_std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32, device=device).view(1, 3, 1, 1)


def preprocess_dino(image: Image.Image) -> torch.Tensor:
    image = image.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.BICUBIC)
    array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).to(device)
    return (tensor - dino_mean) / dino_std


@torch.no_grad()
def extract_dino_feature_map(image_path: Path) -> np.ndarray:
    image = Image.open(image_path).convert('RGB')
    output = dino_model.forward_features(preprocess_dino(image))
    tokens = output['x_norm_patchtokens'] if isinstance(output, dict) else output
    tokens = tokens.squeeze(0).detach().cpu().float()
    grid = int(math.sqrt(tokens.shape[0]))
    if grid * grid != tokens.shape[0]:
        raise ValueError(f'DINO token count is not square: {tokens.shape[0]}')
    return tokens.reshape(grid, grid, tokens.shape[-1]).numpy()

print('DINO backbone ready:', FEATURE_BACKBONE)


## Cell Role: Build Memory Bank

clean memory image의 normalized mask를 patch-bag instance로 변환하고, clean descriptor clustering으로 prototype memory와 prototype-level spatial graph를 만든다.


In [ ]:
from graph_probe.component_io import save_json
from relation.grounding_mask_cluster import raw_masks_from_label_image
from stage1_adapter import (
    CandidateMaskNormalizationConfig,
    MatchingConfig,
    build_memory_bank,
    memory_bank_summary,
    normalize_candidate_masks,
)

MASK_NORMALIZATION_CONFIG = CandidateMaskNormalizationConfig(
    max_mask_area_ratio=0.30,
    min_mask_area_ratio=0.001,
    small_cluster_area_ratio=0.006,
    min_cluster_members=3,
    min_cluster_union_area_ratio=0.004,
    max_cluster_union_area_ratio=0.25,
    max_centroid_dist_ratio=0.10,
    max_bbox_gap_ratio=0.04,
)


def normalized_masks_loader(mask_path: Path, image_shape: tuple[int, int]) -> list[dict]:
    raw_masks = raw_masks_from_label_image(Image.open(mask_path).convert('RGB'))
    normalized_masks, _ = normalize_candidate_masks(
        raw_masks,
        image_shape=image_shape,
        config=MASK_NORMALIZATION_CONFIG,
    )
    return normalized_masks

memory_image_paths = [clean_image_path(image_id) for image_id in MEMORY_SAMPLE_IDS]
memory_mask_paths = [clean_mask_path(image_id) for image_id in MEMORY_SAMPLE_IDS]

memory = build_memory_bank(
    memory_image_paths,
    memory_mask_paths,
    feature_extractor=extract_dino_feature_map,
    raw_masks_loader=normalized_masks_loader,
    image_size=IMAGE_SIZE,
    feature_backbone=FEATURE_BACKBONE,
    max_prototypes=MAX_PROTOTYPES,
    normalization_config=MASK_NORMALIZATION_CONFIG,
    min_pairs_for_spatial_edge=MIN_PAIRS_FOR_SPATIAL_EDGE,
    category=CATEGORY,
)

memory_summary = memory_bank_summary(memory)
save_json(memory.to_metadata_dict(), MEMORY_JSON)
print(f'saved memory summary: {MEMORY_JSON}')
display(pd.DataFrame([memory_summary]))
display(
    pd.DataFrame([
        {
            'prototype_id': prototype_id,
            'num_instances': len(entry.instances),
            'occurrence_rate': entry.occurrence_rate,
            'expected_instance_count': entry.expected_instance_count,
            'within_variance': entry.within_variance,
            'q10': entry.within_sim_quantiles.get(0.10),
            'q20': entry.within_sim_quantiles.get(0.20),
            'q50': entry.within_sim_quantiles.get(0.50),
        }
        for prototype_id, entry in memory.prototypes.items()
    ])
)


## Cell Role: Visualize Memory Spatial Graph

prototype별 clean centroid와 spatial edge의 평균 Δposition을 확인한다. W1에서는 prototype-level spatial graph만 사용한다.


In [ ]:
import matplotlib.pyplot as plt

prototype_centroids = {
    prototype_id: np.mean([instance.centroid_pos for instance in entry.instances], axis=0)
    for prototype_id, entry in memory.prototypes.items()
}
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
for prototype_id, centroid in prototype_centroids.items():
    ax.scatter(centroid[0], centroid[1], s=80)
    ax.text(centroid[0], centroid[1], prototype_id, fontsize=8)
for (proto_i, proto_j), edge in memory.spatial_graph.items():
    start = prototype_centroids.get(proto_i)
    if start is None:
        continue
    delta = edge.delta_pos_mean
    ax.arrow(start[0], start[1], delta[0], delta[1], width=0.002, alpha=0.35, length_includes_head=True)
ax.set_xlim(0, 1)
ax.set_ylim(1, 0)
ax.set_title(f'prototype spatial graph edges={len(memory.spatial_graph)}')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

edge_df = pd.DataFrame([
    {
        'prototype_i': edge.prototype_i,
        'prototype_j': edge.prototype_j,
        'dx_mean': edge.delta_pos_mean[0],
        'dy_mean': edge.delta_pos_mean[1],
        'num_pairs': edge.num_pairs,
    }
    for edge in memory.spatial_graph.values()
])
display(edge_df)


## Cell Role: Clean Self-Probe Inference

memory에 들어가지 않은 clean normal 한 장을 query로 넣는다. Query는 raw DINO patch만 사용하며, query-side SAM mask를 사용하지 않는다.


In [ ]:
from stage1_adapter import (
    decomposition_summary,
    match_query_against_memory,
    query_patch_grid_from_feature_map,
)

matching_config = MatchingConfig(
    soft_assign_quantile=0.20,
    ot_reg=0.05,
    ot_marginal_penalty=1.0,
    min_matched_mass_for_instance=0.5,
    skip_signal_4_if_no_extent=True,
)

query_feature_map = extract_dino_feature_map(clean_image_path(PROBE_SAMPLE_ID))
query_patch_features, query_patch_positions = query_patch_grid_from_feature_map(query_feature_map)
result = match_query_against_memory(
    query_patch_features=query_patch_features,
    query_patch_positions=query_patch_positions,
    memory=memory,
    config=matching_config,
)
save_json(result.to_metadata_dict(), SELF_PROBE_JSON)
print(f'saved self-probe result: {SELF_PROBE_JSON}')
display(pd.DataFrame([decomposition_summary(result)]))
per_proto_df = pd.DataFrame([
    value.to_metadata_dict()
    for value in result.per_prototype.values()
]).drop(columns=['transport_shape', 'debug'], errors='ignore')
display(per_proto_df)


## Cell Role: W1 Acceptance Check

스펙 §5.4 기준으로 clean self-probe sanity check를 자동 평가한다. FAIL이면 아직 W2로 넘어가지 않는다.


In [ ]:
from math import comb


def all_signals_finite(decomposition) -> bool:
    values = [
        decomposition.signal_1_total,
        decomposition.signal_2_total,
        decomposition.signal_3_total,
        decomposition.signal_4_total,
        decomposition.total,
    ]
    values.extend(decomposition.spatial_violations.values())
    for proto_result in decomposition.per_prototype.values():
        values.extend([
            proto_result.query_mass,
            proto_result.memory_mass,
            proto_result.matching_cost,
            proto_result.unmatched_query_mass,
            proto_result.unmatched_memory_mass,
            proto_result.instance_count_diff,
        ])
    return bool(np.all(np.isfinite(np.asarray(values, dtype=np.float64))))

num_proto_pairs = comb(len(memory.prototypes), 2) if len(memory.prototypes) >= 2 else 0
total_query_mass = max(1.0, sum(value.query_mass for value in result.per_prototype.values()))
total_memory_mass = max(1.0, sum(value.memory_mass for value in result.per_prototype.values()))
spatial_values = list(result.spatial_violations.values())
checks = {
    'memory has at least 5 prototypes': len(memory.prototypes) >= 5,
    'every prototype has at least one instance': all(len(p.instances) >= 1 for p in memory.prototypes.values()),
    'spatial graph covers >= 50% of prototype pairs': (len(memory.spatial_graph) / max(1, num_proto_pairs)) >= 0.5,
    'signal_1 ratio < 0.20': result.signal_1_total / total_query_mass < 0.20,
    'signal_2 ratio < 0.20': result.signal_2_total / total_memory_mass < 0.20,
    'signal_3 per prototype avg < 0.5': result.signal_3_total / max(1, len(memory.prototypes)) < 0.5,
    'signal_4 avg < 3.0': (float(np.mean(spatial_values)) < 3.0) if spatial_values else True,
    'no NaN in any signal': all_signals_finite(result),
}
check_df = pd.DataFrame([
    {'check': name, 'passed': bool(passed)}
    for name, passed in checks.items()
])
display(check_df)
if check_df['passed'].all():
    print('W1 PASS: memory build and clean self-probe are ready for W2 spec.')
else:
    print('W1 NOT READY: inspect failed checks before extending to shifted/anomaly evaluation.')
